# Arbitrage Derivative Valuation

This notebook demonstrates the three numerical methods for pricing the arbitrage derivative:

1. **Lattice (Binomial Tree)**
2. **Monte Carlo (Least Squares Method)**
3. **PDE (Finite Differences)**

The derivative is long a high-volatility European call and short a low-volatility European call with same strike/expiry. Early unwinding is allowed (American-style).

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import config
import lattice
import monte_carlo
import pde

## 1. Parameters

In [ ]:
# Base parameters
params = config.BASE_PARAMS
print("Base parameters:")
for k, v in params.items():
    print(f"  {k}: {v}")

# Define a non‑constant sigma_fair function for demonstration
# Example: sigma_fair decreases as spot moves away from strike
def sigma_fair_smile(spot: float) -> float:
    """Volatility smile: higher volatility when spot is far from strike."""
    K = params['K']
    rel = abs(np.log(spot / K))
    return 0.2 + 0.1 * rel  # baseline 20% + 10% per log-distance

# Test the function
spots = np.linspace(50, 150, 5)
for s in spots:
    print(f"sigma_fair({s:.1f}) = {sigma_fair_smile(s):.4f}")

## 2. Lattice Method

In [ ]:
n = 200
val_lat, boundary_lat, flags = lattice.binomial_tree_value(
    n=n,
    S0=params['S0'],
    K=params['K'],
    T=params['T'],
    r=params['r'],
    sigma_spot=params['sigma_spot'],
    sigma_fair_func=sigma_fair_smile,
    sigma_offset=params['sigma_offset'],
)

print(f"Lattice value: {val_lat:.6f}")
print(f"Early exercise boundary (first 10 steps):")
print(boundary_lat[:10])

## 2.1 Binomial Tree Visualization

In [ ]:
def plot_binomial_tree(n_steps=5, S0=100.0, K=100.0, T=1.0, r=0.05, sigma_spot=0.2, sigma_fair_func=None, sigma_offset=0.1):
    """
    Plot a binomial tree (first n_steps) showing spot nodes, derivative values, and exercise decisions.
    """
    if sigma_fair_func is None:
        sigma_fair_func = lambda s: 0.2
    
    # Build tree structure
    dt = T / n_steps
    u = np.exp(sigma_spot * np.sqrt(dt))
    d = 1.0 / u
    q = (np.exp(r * dt) - d) / (u - d)
    
    # Spot grid
    spots = {}
    for i in range(n_steps + 1):
        for j in range(i + 1):
            spots[(i, j)] = S0 * (u ** (i - j)) * (d ** j)
    
    # Compute unwind values at each node
    unwind = {}
    for (i, j), spot in spots.items():
        time_to_expiry = T - i * dt
        vol_low = sigma_fair_func(spot)
        vol_high = vol_low + sigma_offset
        price_low = lattice.black_scholes_call(spot, K, time_to_expiry, r, vol_low)
        price_high = lattice.black_scholes_call(spot, K, time_to_expiry, r, vol_high)
        unwind[(i, j)] = price_high - price_low
    
    # Backward induction for derivative values
    value = {}
    exercise = {}
    # Maturity
    for j in range(n_steps + 1):
        value[(n_steps, j)] = unwind[(n_steps, j)]
        exercise[(n_steps, j)] = True  # always unwind at maturity
    
    # Backward
    for i in range(n_steps - 1, -1, -1):
        for j in range(i + 1):
            continuation = np.exp(-r * dt) * (q * value[(i+1, j)] + (1-q) * value[(i+1, j+1)])
            value[(i, j)] = max(unwind[(i, j)], continuation)
            exercise[(i, j)] = unwind[(i, j)] >= continuation
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Draw edges
    for i in range(n_steps):
        for j in range(i + 1):
            x0 = i
            y0 = spots[(i, j)]
            # Up branch
            x1 = i + 1
            y1 = spots[(i+1, j)]
            ax.plot([x0, x1], [y0, y1], 'k-', alpha=0.3, linewidth=0.8)
            # Down branch
            y2 = spots[(i+1, j+1)]
            ax.plot([x0, x1], [y0, y2], 'k-', alpha=0.3, linewidth=0.8)
    
    # Plot nodes
    node_x = []
    node_y = []
    node_color = []
    node_size = []
    node_value = []
    for (i, j), spot in spots.items():
        node_x.append(i)
        node_y.append(spot)
        # Color: green if exercise optimal, red if wait
        node_color.append('limegreen' if exercise[(i, j)] else 'tomato')
        node_size.append(120)
        node_value.append(value[(i, j)])
    
    sc = ax.scatter(node_x, node_y, c=node_color, s=node_size, edgecolors='black', linewidth=0.8, zorder=10)
    
    # Annotate values
    for idx, (x, y, v) in enumerate(zip(node_x, node_y, node_value)):
        if x <= 3:  # only label first few steps to avoid clutter
            ax.annotate(f"{v:.2f}", (x, y), xytext=(0, 10),
                        textcoords='offset points', ha='center', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    # Labels
    ax.set_xlabel('Time Step', fontsize=12)
    ax.set_ylabel('Spot Price', fontsize=12)
    ax.set_title(f'Binomial Tree (First {n_steps} Steps)\nGreen = Exercise | Red = Wait', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='limegreen', edgecolor='black', label='Exercise (Unwind)'),
        Patch(facecolor='tomato', edgecolor='black', label='Wait (Continue)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.savefig('../output/binomial_tree.png', dpi=150)
    plt.show()
    
    # Print summary
    print(f"Initial value: {value[(0,0)]:.6f}")
    print(f"Unwind at t=0: {unwind[(0,0)]:.6f}")
    print(f"Exercise at root? {exercise[(0,0)]}")
    return value, exercise, spots

In [ ]:
# Plot a small tree with the same parameters
value_dict, exercise_dict, spots_dict = plot_binomial_tree(
    n_steps=5,
    S0=params['S0'],
    K=params['K'],
    T=params['T'],
    r=params['r'],
    sigma_spot=params['sigma_spot'],
    sigma_fair_func=sigma_fair_smile,
    sigma_offset=params['sigma_offset']
)

## 3. Monte Carlo LSM

In [ ]:
n_paths = 10000
n_steps = 50
val_mc, boundary_mc = monte_carlo.lsm_value(
    n_paths=n_paths,
    n_steps=n_steps,
    S0=params['S0'],
    K=params['K'],
    T=params['T'],
    r=params['r'],
    sigma_spot=params['sigma_spot'],
    sigma_fair_func=sigma_fair_smile,
    sigma_offset=params['sigma_offset'],
)

print(f"Monte Carlo LSM value: {val_mc:.6f}")
print(f"Early exercise boundary (first 10 steps):")
print(boundary_mc[:10])

## 4. PDE Method (Placeholder)

In [ ]:
# PDE not fully implemented yet
val_pde, grid_vals, boundary_pde = pde.pde_value(
    S_min=1.0,
    S_max=300.0,
    n_S=200,
    n_t=100,
    S0=params['S0'],
    K=params['K'],
    T=params['T'],
    r=params['r'],
    sigma_spot=params['sigma_spot'],
    sigma_fair_func=sigma_fair_smile,
    sigma_offset=params['sigma_offset'],
)
print(f"PDE value: {val_pde:.6f}")

## 5. Comparison and Visualization

In [ ]:
# Compute European benchmark
from lattice import black_scholes_call
vol_low = sigma_fair_smile(params['S0'])
vol_high = vol_low + params['sigma_offset']
price_low = black_scholes_call(params['S0'], params['K'], params['T'], params['r'], vol_low)
price_high = black_scholes_call(params['S0'], params['K'], params['T'], params['r'], vol_high)
european_diff = price_high - price_low

print(f"European difference (no early exercise): {european_diff:.6f}")
print(f"Early exercise premium (Lattice): {val_lat - european_diff:.6f}")
print(f"Early exercise premium (Monte Carlo): {val_mc - european_diff:.6f}")

# Plot exercise boundaries
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boundary comparison
time_lat = np.arange(len(boundary_lat))
time_mc = np.arange(len(boundary_mc))
axes[0].plot(time_lat, boundary_lat, 'o-', label='Lattice', markersize=3)
axes[0].plot(time_mc, boundary_mc, 's-', label='Monte Carlo', markersize=3)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Spot Boundary')
axes[0].set_title('Exercise Boundary Comparison')
axes[0].legend()
axes[0].grid(True)

# Volatility smile
spots = np.linspace(50, 150, 100)
vols = [sigma_fair_smile(s) for s in spots]
axes[1].plot(spots, vols, 'b-')
axes[1].axvline(params['S0'], color='r', linestyle='--', label=f'S0={params["S0"]}')
axes[1].set_xlabel('Spot')
axes[1].set_ylabel('sigma_fair(spot)')
axes[1].set_title('Volatility Smile')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../output/notebook_results.png', dpi=150)
plt.show()

## 6. Convergence Study

In [ ]:
# Study convergence of lattice as n increases
n_values = [50, 100, 200, 400, 800]
vals = []
for n in n_values:
    val, _, _ = lattice.binomial_tree_value(
        n=n,
        S0=params['S0'],
        K=params['K'],
        T=params['T'],
        r=params['r'],
        sigma_spot=params['sigma_spot'],
        sigma_fair_func=sigma_fair_smile,
        sigma_offset=params['sigma_offset'],
    )
    vals.append(val)
    print(f"n={n:4d}, value={val:.6f}")

plt.figure()
plt.plot(n_values, vals, 'o-')
plt.xlabel('Number of time steps (n)')
plt.ylabel('Derivative Value')
plt.title('Lattice Convergence')
plt.grid(True)
plt.savefig('../output/lattice_convergence.png', dpi=150)
plt.show()

## Summary

The three methods should produce consistent values within tolerance. Early exercise premium depends on the shape of `sigma_fair(spot)`. The PDE method remains to be fully implemented.